1. evaluate a bunch of CNNs with a couple of different metanetworks
2. get intra-CNN variance to analyze how metanetwork-invariant the results are
3. if this is the case, see if we can cluster bad vs. good CNNs for unlearning
4. Analyze why this is

# 0. Train a couple of metanetworks ✅

In [1]:
from cnn_surgery.utils.load_dataset import load_multi_stage_dataset
from cnn_surgery.lenses.regressor_lens import get_regressor_lens
import torch
from tqdm import tqdm
import os
import json
import pickle

datasets = ["mnist", "fashion_mnist", "cifar10"]

for dataset in datasets:
    train, val, _ = load_multi_stage_dataset(include_test=False, dataset=dataset).values() # type: ignore

    weights_train = train[0]
    weights_val = val[0]

    accuracies_train = train[1]
    accuracies_val = val[1]

    configs_train = train[2]
    configs_val = val[2]

    for i in range(5):
        MetaNetwork, metrics = get_regressor_lens(
            weights_train,
            accuracies_train,
            weights_val,
            accuracies_val,
            device="cpu",
            return_metrics=True,
            verbose=False,
        )  # type: ignore

        # make sure parent directory exists
        os.makedirs("../models/good_bad_experiment_2", exist_ok=True)
        torch.save(MetaNetwork.state_dict(), f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}.pt")
        # should've saved them as pickles
        # with open(f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}.pkl", "wb") as f:
        #     pickle.dump(MetaNetwork, f)

        # save metrics
        metrics_dict = {
            "mse_train": metrics[0][0],
            "mae_train": metrics[0][1],
            "mse_val": metrics[1][0],
            "mae_val": metrics[1][1],
            "r2_val": metrics[2],
        }

        with open(f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}_metrics.json", "w") as f:
            json.dump(metrics_dict, f, indent=2)

ModuleNotFoundError: No module named 'cnn_surgery'

## 0.1 Analyze the metanetworks (are they all good?) ✅

plot mean and std of the metanetworks per dataset

In [22]:
import json
import pandas as pd
import plotly.express as px

metrics_df = pd.DataFrame()

datasets = ["mnist", "fashion_mnist", "cifar10"]
for dataset in datasets:
    for i in range(5):
        metrics_path = f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}_metrics.json"
        with open(metrics_path, "r") as f:
            metrics = json.load(f)
            
            # add the json data to the dataframe
            row = {"dataset": dataset, "metanetwork_idx": i} 
            row.update(metrics)
            metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)


display(metrics_df)
metrics_df.describe()

# plot r2_val per dataset (narrower figure)
fig = px.box(metrics_df, x="dataset", y="r2_val", points="all", title="Metanetwork R2 on validation set per dataset")
fig.update_layout(width=500, height=400)
fig.show()

,dataset,metanetwork_idx,mse_train,mae_train,mse_val,mae_val,r2_val
0,mnist,0,0.014684,0.055626,0.016694,0.061677,0.906895
1,mnist,1,0.012287,0.050942,0.014185,0.056650,0.920888
2,mnist,2,0.012260,0.050746,0.014176,0.056628,0.920937
3,mnist,3,0.011747,0.049079,0.013688,0.055090,0.923660
4,mnist,4,0.012120,0.050808,0.014101,0.056968,0.921355
5,fashion_mnist,0,0.012757,0.059341,0.016117,0.068093,0.891528
6,fashion_mnist,1,0.012202,0.058472,0.015308,0.066764,0.896975
7,fashion_mnist,2,0.014176,0.064582,0.017403,0.072770,0.882873
8,fashion_mnist,3,0.012264,0.058795,0.015350,0.067037,0.896690
9,fashion_mnist,4,0.012783,0.060678,0.016129,0.069371,0.891447


# 1. Evaluate a bunch of CNNs with these metanetworks ✅ (200)
# 2. Get intra-CNN variance to analyze how metanetwork-invariant the results are

## 2.1 create a master dataframe ✅

In [32]:
import pandas as pd
import numpy as np
import ast
from cnn_surgery.utils import metrics

datasets = ["mnist", "fashion_mnist", "cifar10"]

master_df = pd.DataFrame()

for dataset in datasets:
    for i in range(5):
        for j in range(10):
            path = f"../experiments/good-bad/{dataset}_eval_results_meta_{i}_class_{j}.csv"
            df = pd.read_csv(path)

            # calculate metrics
            df['max_difference'] = df.apply(lambda row: metrics.max_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class']), axis=1)
            df['clipped_negative_mean_difference'] = df.apply(lambda row: metrics.clipped_negative_mean_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class']), axis=1)
            #df['relative_clipped_negative_mean_difference'] = df.apply(lambda row: metrics.clipped_negative_mean_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class'], proportional=True), axis=1)
            df['target_difference'] = df.apply(lambda row: metrics.target_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class']), axis=1)
            df['mse_init_pred'] = df.apply(lambda row: ((np.array(ast.literal_eval(row['init_pred'])) - np.array(ast.literal_eval(row['original_accuracy'])))**2).mean(), axis=1)
            df['metanetwork_idx'] = i
            df['metanetwork_path'] = f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}.pt"
            df['metanetwork_metrics_path'] = f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}_metrics.json"
            
            master_df = pd.concat([master_df, df], ignore_index=True)

display(master_df)
master_df.describe()


,model_idx,original_accuracy,accuracy_after,overall_accuracy,target_class,dataset,lr,stop_threshold,l2_penalty,loss_fn,...,l2_distance,init_pred,final_pred,max_difference,clipped_negative_mean_difference,target_difference,mse_init_pred,metanetwork_idx,metanetwork_path,metanetwork_metrics_path
0,0,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.1135,0,mnist,0.3,0.5,0.0,boost,...,0.000000,"[0.0141690485, 0.9980428, 0.009339715, 0.00339...","[0.0141690485, 0.9980428, 0.009339715, 0.00339...",0.00000,0.000000,0.000000e+00,0.000163,0,../models/good_bad_experiment_2/mnist_metanetw...,../models/good_bad_experiment_2/mnist_metanetw...
1,1,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.1135,0,mnist,0.3,0.5,0.0,boost,...,0.000000,"[0.013468808, 0.99920243, 0.0062858416, 0.0020...","[0.013468808, 0.99920243, 0.0062858416, 0.0020...",0.00000,0.000000,0.000000e+00,0.000059,0,../models/good_bad_experiment_2/mnist_metanetw...,../models/good_bad_experiment_2/mnist_metanetw...
2,2,"[0.9826531, 0.9920705, 0.9573643, 0.96039605, ...","[0.9918367346938776, 0.9841409691629956, 0.967...",0.9505,0,mnist,0.3,0.5,0.0,boost,...,2.185451,"[0.9850804, 0.9964432, 0.9460311, 0.9522249, 0...","[0.99864894, 0.9998264, 0.9913366, 0.9915206, ...",-0.06859,-0.020637,-9.183635e-03,0.000093,0,../models/good_bad_experiment_2/mnist_metanetw...,../models/good_bad_experiment_2/mnist_metanetw...
3,3,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.1135,0,mnist,0.3,0.5,0.0,boost,...,0.000000,"[0.014699109, 0.99725145, 0.009439481, 0.00404...","[0.014699109, 0.99725145, 0.009439481, 0.00404...",0.00000,0.000000,0.000000e+00,0.000113,0,../models/good_bad_experiment_2/mnist_metanetw...,../models/good_bad_experiment_2/mnist_metanetw...
4,4,"[0.9459184, 0.9929515, 0.96802324, 0.929703, 0...","[0.9459183673469388, 0.9929515418502203, 0.969...",0.9397,0,mnist,0.3,0.5,0.0,boost,...,2.113742,"[0.98132026, 0.9958727, 0.9435033, 0.9542402, ...","[0.99840754, 0.99979, 0.99155575, 0.99279046, ...",-0.06323,-0.021210,3.265306e-08,0.000387,0,../models/good_bad_experiment_2/mnist_metanetw...,../models/good_bad_experiment_2/mnist_metanetw...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,195,"[0.493, 0.518, 0.285, 0.135, 0.315, 0.748, 0.7...","[0.567, 0.559, 0.238, 0.142, 0.316, 0.748, 0.6...",0.4909,9,cifar10,0.3,0.5,0.0,boost,...,1.121220,"[0.58511204, 0.59940565, 0.29700574, 0.1342383...","[0.60940224, 0.61098254, 0.31371495, 0.1362241...",0.04200,0.104333,1.180000e-01,0.007568,4,../models/good_bad_experiment_2/cifar10_metane...,../models/good_bad_experiment_2/cifar10_metane...
29996,196,"[0.526, 0.648, 0.26, 0.059, 0.507, 0.454, 0.67...","[0.526, 0.648, 0.26, 0.059, 0.507, 0.454, 0.67...",0.4499,9,cifar10,0.3,0.5,0.0,boost,...,0.000000,"[0.42242387, 0.381152, 0.26006863, 0.08809308,...","[0.42242387, 0.381152, 0.26006863, 0.08809308,...",0.00000,0.000000,0.000000e+00,0.025905,4,../models/good_bad_experiment_2/cifar10_metane...,../models/good_bad_experiment_2/cifar10_metane...
29997,197,"[0.387, 0.525, 0.201, 0.104, 0.304, 0.489, 0.4...","[0.387, 0.525, 0.201, 0.104, 0.304, 0.489, 0.4...",0.3895,9,cifar10,0.3,0.5,0.0,boost,...,0.000000,"[0.43531984, 0.4668035, 0.23087603, 0.1252136,...","[0.43531984, 0.4668035, 0.23087603, 0.1252136,...",0.00000,0.000000,0.000000e+00,0.003183,4,../models/good_bad_experiment_2/cifar10_metane...,../models/good_bad_experiment_2/cifar10_metane...
29998,198,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.1000,9,cifar10,0.3,0.5,0.0,boost,...,0.000000,"[0.97876066, 0.016933637, 0.0127763115, 0.0016...","[0.97876066, 0.016933637, 0.0127763115, 0.0016...",0.00000,0.000000,0.000000e+00,0.000397,4,../models/good_bad_experiment_2/cifar10_metane...,../models/good_bad_experiment_2/cifar10_

,model_idx,overall_accuracy,target_class,lr,stop_threshold,l2_penalty,max_steps,steps,final_loss,distance_travelled,l2_distance,max_difference,clipped_negative_mean_difference,target_difference,mse_init_pred,metanetwork_idx
count,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.0,30000.0,30000.0,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.000000,30000.000000,30000.000000
mean,99.500000,0.472415,4.500000,3.000000e-01,0.5,0.0,1000.0,243.258433,-0.059473,1.321909,1.193003,-0.080569,5.082259e-02,0.116850,0.010090,2.000000
std,57.735267,0.270003,2.872329,5.551208e-17,0.0,0.0,0.0,389.639601,0.168304,1.790307,1.621747,0.250863,1.910563e-01,0.241943,0.021629,1.414237
min,0.000000,0.035800,0.000000,3.000000e-01,0.5,0.0,1000.0,0.000000,-0.707865,0.000000,0.000000,-1.455000,-8.953092e-01,-0.707000,0.000021,0.000000
25%,49.750000,0.233300,2.000000,3.000000e-01,0.5,0.0,1000.0,0.000000,-0.161981,0.000000,0.000000,-0.114893,-1.331604e-09,0.000000,0.000583,1.000000
50%,99.500000,0.434400,4.500000,3.000000e-01,0.5,0.0,1000.0,2.000000,-0.082905,0.296915,0.294282,0.000000,0.000000e+00,0.000000,0.002296,2.000000
75%,149.250000,0.706700,7.000000,3.000000e-01,0.5,0.0,1000.0,345.250000,0.085685,2.424833,2.195970,0.000000,2.466667e-02,0.097000,0.008215,3.000000
max,199.000000,0.984300,9.000000,3.000000e-01,0.5,0.0,1000.0,999.000000,0.457471,49.709743,49.709743,1.000000,1.000000e+00,1.000000,0.199774,4.000000


In [38]:
import plotly.express as px

# drop all rows with dataset mnist or cifar10
master_df = master_df[master_df['dataset'] != 'mnist']
master_df = master_df[master_df['dataset'] != 'cifar10']

# only use rows with top 10% average initial accuracy
master_df['avg_initial_accuracy'] = master_df['original_accuracy'].apply(lambda x: np.mean(ast.literal_eval(x)))
master_df = master_df[master_df['avg_initial_accuracy'] >= master_df['avg_initial_accuracy'].quantile(0.9)]

# Group by model_idx, dataset, target_class and calculate mean and std over metanetwork_idx
grouped_stats = master_df.groupby(['model_idx', 'dataset', 'target_class']).agg({
    'max_difference': ['mean', 'std'],
    'clipped_negative_mean_difference': ['mean', 'std'],
    'target_difference': ['mean', 'std']
}).reset_index()

# Flatten column names
grouped_stats.columns = ['_'.join(col).strip('_') for col in grouped_stats.columns]
display(grouped_stats)

# Melt the dataframe to get std columns in long format for plotting
std_cols = ['max_difference_std', 'clipped_negative_mean_difference_std', 'target_difference_std']
melted_std = grouped_stats.melt(
    id_vars=['model_idx', 'dataset', 'target_class'],
    value_vars=std_cols,
    var_name='metric',
    value_name='std_value'
)

## STDs
# Clean up metric names for display
melted_std['metric'] = melted_std['metric'].str.replace('_std', '')

# Create boxplot
fig = px.scatter(melted_std, x='metric', y='std_value', color='dataset', 
             title='Standard Deviation of Metrics Across Metanetworks',
             labels={'std_value': 'Standard Deviation', 'metric': 'Metric'})
fig.show()

## means
# Melt the dataframe to get mean columns in long format for plotting
mean_cols = ['max_difference_mean', 'clipped_negative_mean_difference_mean', 'target_difference_mean']
melted_mean = grouped_stats.melt(
    id_vars=['model_idx', 'dataset', 'target_class'],
    value_vars=mean_cols,
    var_name='metric',
    value_name='mean_value'
)

# Clean up metric names for display
melted_mean['metric'] = melted_mean['metric'].str.replace('_mean', '')

# Create boxplot
fig = px.scatter(melted_mean, x='metric', y='mean_value', color='dataset', 
             title='Mean of Metrics Across Metanetworks',
             labels={'mean_value': 'Mean', 'metric': 'Metric'})
fig.show()

## Detailed boxplots per metanetwork
# Plot max_difference per metanetwork (5 metanetworks x 3 datasets = 15 boxplots)
# fig = px.scatter(master_df[master_df['model_idx'] == 3][master_df['target_class'] == 1], x='metanetwork_idx', y='max_difference', color='model_idx',
#              title='Max Difference per Metanetwork',
#              labels={'max_difference': 'Max Difference', 'metanetwork_idx': 'Metanetwork Index'},
#              hover_data=['target_class'])
# fig.update_yaxes(range=[-1, 1])
# fig.show()

# # Plot clipped_negative_mean_difference per metanetwork
# fig = px.scatter(master_df[master_df['target_class'] == 1], x='metanetwork_idx', y='clipped_negative_mean_difference', color='dataset',
#              title='Clipped Negative Mean Difference per Metanetwork',
#              labels={'clipped_negative_mean_difference': 'Clipped Negative Mean Difference', 'metanetwork_idx': 'Metanetwork Index'},
#              hover_data=['target_class'],
#              animation_frame='model_idx')
# fig.update_yaxes(range=[-1, 1])
# fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 100
# fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 0
# fig.show()

# # Plot target_difference per metanetwork
# fig = px.scatter(master_df[master_df['model_idx'] == 3][master_df['target_class'] == 1], x='metanetwork_idx', y='target_difference', color='model_idx',
#              title='Target Difference per Metanetwork',
#              labels={'target_difference': 'Target Difference', 'metanetwork_idx': 'Metanetwork Index'},
#              hover_data=['target_class'])
# fig.update_yaxes(range=[-1, 1])
# fig.show()

# ## Quantify the difference in performance (expressed in the three metrics) of the five metanetworks
# # Calculate mean performance metrics per metanetwork and dataset
# metanetwork_performance = master_df.groupby(['metanetwork_idx', 'dataset']).agg({
#     'max_difference': 'std',
#     'clipped_negative_mean_difference': 'std',
#     'target_difference': 'std'
# }).reset_index()

# display(metanetwork_performance)

# # Calculate summary statistics across metanetworks per dataset
# print("\nPerformance variation across metanetworks per dataset:")
# performance_summary = metanetwork_performance.groupby('dataset').agg({
#     'max_difference': ['mean', 'std', 'min', 'max'],
#     'clipped_negative_mean_difference': ['mean', 'std', 'min', 'max'],
#     'target_difference': ['mean', 'std', 'min', 'max']
# })
# display(performance_summary)

# # ANOVA-style analysis: variance between metanetworks vs within
# print("\nCoefficient of Variation (std/|mean|) across metanetworks per dataset:")
# cv_stats = metanetwork_performance.groupby('dataset').apply(
#     lambda x: pd.Series({
#         'max_difference_cv': x['max_difference'].std() / abs(x['max_difference'].mean()) if x['max_difference'].mean() != 0 else 0,
#         'clipped_negative_mean_difference_cv': x['clipped_negative_mean_difference'].std() / abs(x['clipped_negative_mean_difference'].mean()) if x['clipped_negative_mean_difference'].mean() != 0 else 0,
#         'target_difference_cv': x['target_difference'].std() / abs(x['target_difference'].mean()) if x['target_difference'].mean() != 0 else 0
#     })
# )
# display(cv_stats)

,model_idx,dataset,target_class,max_difference_mean,max_difference_std,clipped_negative_mean_difference_mean,clipped_negative_mean_difference_std,target_difference_mean,target_difference_std
0,26,fashion_mnist,0,-0.2558,0.324440,0.111711,0.167846,0.2636,0.142744
1,26,fashion_mnist,1,-0.3870,0.046481,-0.110511,0.043307,0.0108,0.006834
2,26,fashion_mnist,2,-0.0932,0.095492,0.057267,0.052892,0.1018,0.055903
3,26,fashion_mnist,3,-0.1222,0.054646,-0.017644,0.010773,0.0128,0.007855
4,26,fashion_mnist,4,0.1044,0.021396,0.172689,0.030411,0.1894,0.033872
5,26,fashion_mnist,5,-0.1462,0.072064,-0.017200,0.037521,0.0096,0.042595
6,26,fashion_mnist,6,0.0960,0.024280,0.101289,0.023706,0.1026,0.023607
7,26,fashion_mnist,7,-0.0898,0.057129,0.064889,0.022066,0.1016,0.026197
8,26,fashion_mnist,8,-0.3280,0.034778,-0.070756,0.031361,0.0366,0.012779
9,26,fashion_mnist,9,-0.3368,0.078960,-0.086000,0.024243,-0.0088,0.008438


What are characteristics of the outliers?
gemiddelde performance per metanetwerk